## **META Reasoning Example**

### Simulated travel agent with meta-reasoning capabilities

In [1]:
import random 

class ReflectiveTravelAgent:

    def __init__(self):
        # Initialize preference weights
        self.preferences_weights = {
            "budget": 0.5,
            "luxury": 0.3,
            "adventure": 0.2
        }
        self.user_feedback = []  
        
        # Destination penalty/reward system
        self.destination_adjustments = {
            "Paris": 0.0,
            "Bangkok": 0.0,
            "New York": 0.0
        }

    def recommended_destination(self, user_preferences):
        """
        Recommend a destination based on user preferences, weights, and adjustments.
        """
        score = {
            'Paris': (self.preferences_weights["luxury"] * user_preferences["luxury"] +
                      self.preferences_weights["adventure"] * user_preferences["adventure"]) 
                      + self.destination_adjustments["Paris"],

            'Bangkok': (self.preferences_weights["budget"] * user_preferences["budget"] +
                        self.preferences_weights["adventure"] * user_preferences["adventure"]) 
                        + self.destination_adjustments["Bangkok"],

            'New York': (self.preferences_weights["luxury"] * user_preferences["luxury"] +
                         self.preferences_weights["budget"] * user_preferences["budget"]) 
                         + self.destination_adjustments["New York"]
        }

        recommendation = max(score, key=score.get)
        return recommendation

    def get_user_feedback(self, actual_experience):
        """
        Simulate receiving user feedback and trigger meta-reasoning.
        """
        feedback = random.choice([1, -1])  # 1 = positive, -1 = negative
        print(f"Feedback for {actual_experience}: {'Positive' if feedback == 1 else 'Negative'}")

        self.user_feedback.append((actual_experience, feedback))
        self.meta_reasoning()

    def meta_reasoning(self):
        """
        Adjust weights and destination adjustments based on feedback.
        """
        for destination, feedback in self.user_feedback:
            if feedback == -1:
                # Penalize destination directly
                self.destination_adjustments[destination] -= 0.2
                # Reduce main attribute weight more strongly
                if destination == "Paris":
                    self.preferences_weights["luxury"] *= 0.8
                elif destination == "Bangkok":
                    self.preferences_weights["budget"] *= 0.8
                elif destination == "New York":
                    self.preferences_weights["budget"] *= 0.8
            elif feedback == 1:
                # Reward destination directly
                self.destination_adjustments[destination] += 0.2
                # Increase main attribute weight
                if destination == "Paris":
                    self.preferences_weights["luxury"] *= 1.2
                elif destination == "Bangkok":
                    self.preferences_weights["budget"] *= 1.2
                elif destination == "New York":
                    self.preferences_weights["budget"] *= 1.2

        # Normalize weights
        total_weight = sum(self.preferences_weights.values())
        for key in self.preferences_weights:
            self.preferences_weights[key] /= total_weight

        print(f"Updated weights: {self.preferences_weights}")
        print(f"Destination adjustments: {self.destination_adjustments}\n")


### Simulation

In [2]:
agent = ReflectiveTravelAgent()

# User's initial preferences
user_preferences = {
        "budget": 0.2,      # High preference for budget-friendly options
        "luxury": 0.8,      # Low preference for luxury
        "adventure": 0.5    # Moderate preference for adventure activities
}

# First Recommendation based on the initial preferences
recommended = agent.recommended_destination(user_preferences)
print(f"Recommended Destination :- {recommended}")

# Simulate user experience and provide feedback
agent.get_user_feedback(recommended)

# Second recommendation based on the internal weight update
recommended = agent.recommended_destination(user_preferences) 
print(f"Updated recommendation: {recommended}")

Recommended Destination :- Paris
Feedback for Paris: Negative
Updated weights: {'budget': 0.5319148936170213, 'luxury': 0.2553191489361702, 'adventure': 0.21276595744680854}
Destination adjustments: {'Paris': -0.2, 'Bangkok': 0.0, 'New York': 0.0}

Updated recommendation: New York


## **META Reasoning with AI**

In [3]:
import getpass
import openai , os 

api_key = getpass.getpass(prompt = "Enter OpenAI API :- ")
os.environ['OPENAI_API_KEY'] = api_key

In [4]:
from crewai.tools import tool

# Tool 1
@tool("Recommend travel destination based on preferences.")
def recommend_destination(user_preferences: dict) -> str:
    """
    Recommend a destination based on user preferences and internal weightings.

    Args:
        user_preferences (dict): User's preferences with keys - 'budget', 'luxury', 'adventure'
                                default user_preference weights 'budget' = 0.8, 'luxury' = 0.2, 'adventure' = 0.5
                                user_preferences = {
                                                "budget": 0.8,
                                                "luxury": 0.4,
                                                "adventure": 0.3
                                            }
    Returns:
        str: Recommended destination
    """
    internal_default_weights = {
            "budget": 0.33,    # Weight for budget-related preferences
            "luxury": 0.33,    # Weight for luxury-related preferences
            "adventure": 0.33  # Weight for adventure-related preferences
        }
   # Calculate weighted scores for each destination
    score = {
        "Paris": (
            internal_default_weights["luxury"] * user_preferences["luxury"] +      # Paris emphasizes luxury
            internal_default_weights["adventure"] * user_preferences["adventure"] +
            internal_default_weights["budget"] * user_preferences["budget"]
        ),
        "Bangkok": (
            internal_default_weights["budget"] * user_preferences["budget"] * 2 +  # Bangkok emphasizes budget
            internal_default_weights["luxury"] * user_preferences["luxury"] +
            internal_default_weights["adventure"] * user_preferences["adventure"]
        ),
        "New York": (
            internal_default_weights["luxury"] * user_preferences["luxury"] * 1.5 +  # NYC emphasizes luxury and adventure
            internal_default_weights["adventure"] * user_preferences["adventure"] * 1.5 +
            internal_default_weights["budget"] * user_preferences["budget"]
        )
    }
    
    # Select the destination with the highest calculated score
    recommendation = max(score, key=score.get)
    return recommendation

# Tool 2
@tool("Reasoning tool to adjust preference weights based on user feedback.")
def update_weights_on_feedback(destination: str, feedback: int, adjustment_factor: float) -> dict:
    """
    Analyze collected feedback and adjust internal preference weights based on user feedback for better future recommendations.

    Args:        
        destination (str): The destination recommended ('New York', 'Bangkok' or 'Paris')
        feedback (int): Feedback score; 1 = Satisfied, -1 = dissatisfied
        adjustment_factor (int): The adjustment factor between 0 and 1 that will be used to adjust the internal weights.
                                 Value will be used as (1 - adjustment_factor) for dissatisfied feedback and (1 + adjustment_factor)
                                 for satisfied feedback.
    Returns:
        dict: Adjusted internal weights
    """
    internal_default_weights = {
        "budget": 0.33,    # Weight for budget-related preferences
        "luxury": 0.33,    # Weight for luxury-related preferences
        "adventure": 0.33  # Weight for adventure-related preferences
    }

    # Define primary and secondary characteristics for each destination
    destination_characteristics = {
        "Paris": {
            "primary": "luxury",
            "secondary": "adventure"
        },
        "Bangkok": {
            "primary": "budget",
            "secondary": "adventure"
        },
        "New York": {
            "primary": "luxury",
            "secondary": "adventure"
        }
    }

    # Get the characteristics for the given destination
    dest_chars = destination_characteristics.get(destination, {})
    primary_feature = dest_chars.get("primary")
    secondary_feature = dest_chars.get("secondary")

    # adjustment_factor = 0.2  # How much to adjust weights by

    if feedback == -1:  # Negative feedback
        # Decrease weights for the destination's characteristics
        if primary_feature:
            internal_default_weights[primary_feature] *= (1 - adjustment_factor)
        if secondary_feature:
            internal_default_weights[secondary_feature] *= (1 - adjustment_factor/2)
            
    elif feedback == 1:  # Positive feedback
        # Increase weights for the destination's characteristics
        if primary_feature:
            internal_default_weights[primary_feature] *= (1 + adjustment_factor)
        if secondary_feature:
            internal_default_weights[secondary_feature] *= (1 + adjustment_factor/2)

    # Normalize weights to ensure they sum up to 1
    total_weight = sum(internal_default_weights.values())
    for key in internal_default_weights:
        internal_default_weights[key] = round(internal_default_weights[key] / total_weight, 2)

    # Ensure weights sum to exactly 1.0 after rounding
    adjustment = 1.0 - sum(internal_default_weights.values())
    if adjustment != 0:
        # Add any rounding difference to the largest weight
        max_key = max(internal_default_weights, key=internal_default_weights.get)
        internal_default_weights[max_key] = round(internal_default_weights[max_key] + adjustment, 2)

    return internal_default_weights

# Tool 3
@tool("User feedback emulator tool")
def feedback_emulator(destination: str) -> int:
    """
    Given a destination recommendation (such as 'New York' or 'Bangkok') this tool will emulate to provide
    a user feedback as 1 (satisfied) or -1 (dissatisfied)
    """
    import random
    feedback = random.choice([-1, 1])
    return feedback

In [ ]:
from crewai import Agent, Task, Crew
from typing import Dict, Union
import random

# Utility functions
def process_recommendation_output(output: str) -> str:
    """Extract the clean destination string from the agent's output."""
    # Handle various ways the agent might format the destination
    for city in ["Paris", "Bangkok", "New York"]:
        if city.lower() in output.lower():
            return city
    return output.strip()

def process_feedback_output(output: Union[Dict, str]) -> int:
    """Extract the feedback value from the agent's output."""
    if isinstance(output, dict):
        return output.get('feedback', 0)
    try:
        # Try to parse as integer if it's a string
        return int(output)
    except (ValueError, TypeError):
        return 0

def generate_random_preferences():
    # Generate 3 random numbers and normalize them
    values = [random.random() for _ in range(3)]
    total = sum(values)
    
    return {
        "budget": round(values[0]/total, 2),
        "luxury": round(values[1]/total, 2),
        "adventure": round(values[2]/total, 2)
    }

# Initial shared state for weights, preferences, and results
state = {
    "weights": {"budget": 0.33, "luxury": 0.33, "adventure": 0.33},
    "preferences": generate_random_preferences()
}

# Agents
preference_agent = Agent(
    name="Preference Agent",
    role="Travel destination recommender",
    goal="Provide the best travel destination based on user preferences and weights.",
    backstory="An AI travel expert adept at understanding user preferences.",
    verbose=True,
    llm='gpt-4o-mini',
    tools=[recommend_destination]
)

feedback_agent = Agent(
    name="Feedback Agent",
    role="Simulated feedback provider",
    goal="Provide simulated feedback for the recommended travel destination.",
    backstory="An AI that mimics user satisfaction or dissatisfaction for travel recommendations.",
    verbose=True,
    llm='gpt-4o-mini',
    tools=[feedback_emulator]
)

meta_agent = Agent(
    name="Meta-Reasoning Agent",
    role="Preference weight adjuster",
    goal="Reflect on feedback and adjust the preference weights to improve future recommendations.",
    backstory="An AI optimizer that learns from user experiences to fine-tune recommendation preferences.",
    verbose=True,
    llm='gpt-4o-mini',
    tools=[update_weights_on_feedback]
)


# Tasks with data passing
generate_recommendation = Task(
    name="Generate Recommendation",
    agent=preference_agent,
    description=(
        f"Use the recommend_destination tool with these preferences: {state['preferences']}\n"
        "Return only the destination name as a simple string (Paris, Bangkok, or New York)."
    ),
    expected_output="A destination name as a string",
    output_handler=process_recommendation_output
)

simulate_feedback = Task(
    name="Simulate User Feedback",
    agent=feedback_agent,
    description=(
        "Use the feedback_emulator tool with the destination from the previous task.\n"
        "Instructions:\n"
        "1. Get the destination string from the previous task\n"
        "2. Pass it directly to the feedback_emulator tool\n"
        "3. Return the feedback value (1 or -1)\n\n"
        "IMPORTANT: Pass the destination as a plain string, not a dictionary."
    ),
    expected_output="An integer feedback value: 1 or -1",
    context=[generate_recommendation],
    output_handler=process_feedback_output
)

adjust_weights = Task(
    name="Adjust Weights Based on Feedback",
    agent=meta_agent,
    description=(
        "Use the update_weights_on_feedback tool with:\n"
        "1. destination: Get from first task's output (context[0])\n"
        "2. feedback: Get from second task's output (context[1])\n"
        "3. adjustment_factor: a number betweek 0 and 1 that will be used to adjust internal weights based on feedback\n\n"
        "Ensure all inputs are in their correct types (string for destination, integer for feedback)."
    ),
    expected_output="Updated weights as a dictionary",
    context=[generate_recommendation, simulate_feedback]
)

# Crew Definition
crew = Crew(
    agents=[preference_agent, feedback_agent, meta_agent],
    tasks=[generate_recommendation, simulate_feedback, adjust_weights],
    verbose=False
)

# Execute the workflow
result = await crew.kickoff_async()
print("\nFinal Results:", result)

╭─────────────────────────────────────────────── 🤖 Agent Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Travel destination recommender                                                                          │
│                                                                                                                 │
│  Task: Use the recommend_destination tool with these preferences: {'budget': 0.34, 'luxury': 0.41,              │
│  'adventure': 0.24}                                                                                             │
│  Return only the destination name as a simple string (Paris, Bangkok, or New York).                             │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

Tool recommend_travel_destination_based_on_preferences executed with result: Error executing tool: 'luxury'...
Tool recommend_travel_destination_based_on_preferences executed with result: Error executing tool: 'luxury'...
Tool recommend_travel_destination_based_on_preferences executed with result: Error executing tool: 'luxury'...
Tool recommend_travel_destination_based_on_preferences executed with result: Error executing tool: 'luxury'...
Tool recommend_travel_destination_based_on_preferences executed with result: Error executing tool: 'luxury'...
Tool recommend_travel_destination_based_on_preferences executed with result: Error executing tool: 'luxury'...
Tool recommend_travel_destination_based_on_preferences executed with result: Error executing tool: 'luxury'...
Tool recommend_travel_destination_based_on_preferences executed with result: Error executing tool: 'luxury'...
Tool recommend_travel_destination_based_on_preferences executed with result: Error executing tool: 'luxury'...
T